# 01 - Setup y Autenticacion GEE
## Proyecto Boomerang - SpaceHACK 2026

Este notebook configura el entorno de trabajo con Google Earth Engine via geemap.

In [1]:
 !pip install geemap earthengine-api


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import ee
import geemap

# Autenticacion - ejecutar la primera vez (abre el navegador)
# ee.Authenticate()

# Proyecto GEE: My Project 29952
ee.Initialize(project='august-tower-470819-s6')

print('GEE inicializado correctamente - Proyecto: august-tower-470819-s6')

GEE inicializado correctamente - Proyecto: august-tower-470819-s6


In [4]:
# --- ROI: Golfo de Guayaquil ---
roi = ee.Geometry.Polygon([[
    [-80.23433322304784, -2.581044464678974],
    [-80.09837741250097, -2.581044464678974],
    [-80.09837741250097, -2.4212085241315013],
    [-80.23433322304784, -2.4212085241315013],
    [-80.23433322304784, -2.581044464678974]
]])

Map = geemap.Map(center=[-2.50, -80.17], zoom=12)
Map.addLayer(roi, {'color': 'red'}, 'ROI')
Map

Map(center=[-2.5, -80.17], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [6]:
# --- Verificar carga de Sentinel-2 sobre el ROI ---
s2_test = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi)
    .filterDate('2024-01-01', '2024-12-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
    .median()
    .clip(roi))

vis_rgb = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}
Map.addLayer(s2_test, vis_rgb, 'Sentinel-2 2024 RGB')

print(f'Bandas disponibles: {s2_test.bandNames().getInfo()}')

Bandas disponibles: ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB', 'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE']


In [7]:
# --- Verificar carga de Landsat 8 sobre el ROI ---
l8_test = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    .filterBounds(roi)
    .filterDate('2024-01-01', '2024-12-31')
    .filter(ee.Filter.lt('CLOUD_COVER', 20))
    .median()
    .clip(roi))

def scale_landsat(image):
    optical = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    return image.addBands(optical, overwrite=True)

l8_scaled = scale_landsat(l8_test)
vis_l8 = {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': 0, 'max': 0.3}
Map.addLayer(l8_scaled, vis_l8, 'Landsat 8 2024 RGB')

print(f'Bandas Landsat 8: {l8_test.bandNames().getInfo()}')

Bandas Landsat 8: ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'SR_QA_AEROSOL', 'ST_B10', 'ST_ATRAN', 'ST_CDIST', 'ST_DRAD', 'ST_EMIS', 'ST_EMSD', 'ST_QA', 'ST_TRAD', 'ST_URAD', 'QA_PIXEL', 'QA_RADSAT']


In [8]:
# --- Verificar Landsat 5 historico (2008) ---
l5_test = (ee.ImageCollection('LANDSAT/LT05/C02/T1_L2')
    .filterBounds(roi)
    .filterDate('2008-01-01', '2008-12-31')
    .filter(ee.Filter.lt('CLOUD_COVER', 20)))

print(f'Imagenes Landsat 5 en 2008: {l5_test.size().getInfo()}')
print('Setup completo - todos los sensores verificados sobre el ROI')

Imagenes Landsat 5 en 2008: 0
Setup completo - todos los sensores verificados sobre el ROI
